# Artilharia do campeonato
## Etapa de construção de tabela gold

Agrupado dos artilheiros por jogador e time. Gols contras (o.g.) são desconsiderados.

In [0]:
from pyspark.sql.functions import *
from datetime import datetime, timedelta
from pyspark.sql.window import Window

In [0]:
df_goals= spark.read.table("apifootball.silver.goals")

In [0]:
window_spec = Window.orderBy(desc("goals"))

df_scorers = (
    df_goals
    .filter(~col("player_name").like("%o.g.%"))
    .groupBy("player_name", "team_id","team_name")
    .agg(count("*").alias("goals"))
    .withColumn("rank", rank().over(window_spec))
     .select(
         "rank",
         "player_name",
         "goals",
         "team_id",
         "team_name",
         expr('current_timestamp() - INTERVAL 3 HOURS').alias('datetime_processing_br')
     )
).orderBy(col("goals").desc())

In [0]:
df_scorers.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("apifootball.gold.scorers")

In [0]:
spark.read.table("apifootball.gold.scorers").display()